# Does the rank encoding carry more clinical information than intensity? (Danenberg + Jackson IMC)

**Question.** Rank (each cell's markers ordered from highest to lowest) gives higher kNN enrichment of marker-high cells but a lower same-phenotype share than intensity (`RankTradeoffMarkers`). Does that difference matter for patients?

**Setting (the CyTOF use case).** Only dissociated single-cell profiles are used, no spatial information. Each encoding defines cell states without labels (k-means). A patient is then described by the share of their cells in each state. The same models, folds and patients are used for both encodings.

**Encodings** (full panel: 36 Danenberg / 32 Jackson):
- `int_img`: robust z per image, clipped ±5, as in all earlier runs and the CyTOF code;
- `rank_img`: rank vector of `int_img`;
- `int_coh`, `rank_coh`: the same with one robust z per marker over the whole cohort. This keeps absolute between-patient differences (for example ER level), which per-image z removes.

**Patient features:**
- cell-state composition: k-means (k = 20, 40; 3 seeds; fitted on 200k random cells, then applied to all cells). Features are the CLR of each patient's state shares;
- reference feature sets, not encodings: the published phenotypes' composition, and pseudo-bulk (the patient's mean arcsinh intensity per marker);
- tumour-cell heterogeneity: k-means on epithelial cells only (k = 10, 20). Features are the Shannon entropy of each patient's tumour-state shares, and the same for the published epithelial phenotypes.

**Tests (fixed before the run):**
- **T1, classification:** L2 logistic regression (C = 0.1) on standardised features. 5 × 5-fold stratified cross-validation over patients, with the same folds for every feature set. The out-of-fold probabilities are averaged over repeats and k-means seeds, and scored by AUC (macro one-vs-rest for multiclass).
  - Danenberg targets: PAM50 + claudin-low, IntClust, ER, PR, HER2, grade 3 vs 1–2, lymph-node positive.
  - Jackson targets: clinical type, grade 3, ER, PR, HER2.
- **T2, survival:** penalised Cox (penalty 0.1) on the first 10 PCs of the features, alone and added to clinical covariates. The C-index is computed on out-of-fold risk.
  - Danenberg: disease-specific survival (DSS) and relapse-free survival (RFS). Covariates: age, grade, log tumour size, log positive nodes, ER.
  - Jackson (Basel only): DSS and overall survival. Covariates: age, grade, log tumour size, N+.
- **T3, heterogeneity (the "rank exposes tumour heterogeneity" idea):** Cox for DSS on tumour-state entropy (per SD), adjusted for the clinical covariates and log(number of tumour cells). Also the Spearman correlation with grade and a Kruskal–Wallis test across subtypes.
- **M1, mechanism:** how patient-specific the states are. This is the share of cells in states where one patient contributes more than 50% of the cells.
- **Comparisons:** rank − intensity at the same normalisation and k, with a paired patient-bootstrap 95% CI (1000 resamples; 300 for hazard ratios).

**Primary setting:** `img` normalisation, k = 20 (T1/T2); `img`, k = 10 (T3).

**Decision rules (fixed before the run):**
- **Utility shown** if, in the primary setting, rank − intensity has a CI above 0 for at least half the T1 targets in Danenberg and the same sign in Jackson, **or** the C-index gain for DSS (features + clinical) has a CI above 0 in both cohorts.
- **Heterogeneity supported** if the rank tumour-state entropy has a larger adjusted DSS log-HR than intensity (CI of the difference excluding 0) in Danenberg, with the same sign in Jackson.
- Otherwise the result is reported as "no clinical utility shown". Every target is reported, not only the significant ones.

**Caveat:** k-means is fitted on all patients' cells without labels (transductive but label-free). The folds are used only for the supervised step.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings
import numpy as np, pandas as pd
import pyarrow.parquet as pq
from joblib import Parallel, delayed
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, kruskal
try:
    import lifelines
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lifelines"], check=True)
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 20260929
OUT = "/kaggle/working"
COFACTOR = 1.0
MIN_CELLS = 200             # per image, as in earlier runs
KS_ALL, KS_TUM = (20, 40), (10, 20)
SEEDS = (0, 1, 2)
N_FIT = 200_000
N_REP, N_FOLD = 5, 5
C_LOGIT = 0.1
N_PCA, COX_PEN = 10, 0.1
N_BOOT, N_BOOT_HR = 1000, 300
MIN_TUM = 50                # tumour cells per patient for entropy
MIN_CLASS = 10              # patients per class; smaller classes are dropped
N_JOBS = os.cpu_count()
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
SPACES = ("int_img", "rank_img", "int_coh", "rank_coh")
PAIRS = [("int_img", "rank_img"), ("int_coh", "rank_coh")]

def num(s):
    return pd.to_numeric(pd.Series(s).replace({"nan": np.nan, "[]": np.nan}), errors="coerce")

def locate():
    reps = sorted(glob.glob("/kaggle/input/**/prep_report.json", recursive=True))
    out = {}
    for p in sorted(glob.glob("/kaggle/input/**/cells.parquet", recursive=True)):
        key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
        assert key not in out, (key, p, out)
        rep = max(reps, key=lambda r: len(os.path.commonpath([r, p])))
        clin = max(glob.glob("/kaggle/input/**/clinical.parquet", recursive=True),
                   key=lambda c: len(os.path.commonpath([c, p])))
        out[key] = dict(cells=p, report=rep, clinical=clin)
        if key == "danenberg":
            out[key]["split"] = max(glob.glob("/kaggle/input/**/danenberg_patient_split.csv", recursive=True),
                                    key=lambda c: len(os.path.commonpath([c, p])))
    print(json.dumps(out, indent=1))
    assert set(out) == {"danenberg", "jackson"}
    return out

## Functions: encodings and clinical tables

In [ ]:
def rank_vec(Zs):
    """0 = highest; ties by column order (stable sort), as in the CyTOF code."""
    order = np.argsort(-Zs, axis=1, kind="stable")
    R = np.empty_like(order)
    R[np.arange(len(Zs))[:, None], order] = np.arange(Zs.shape[1])
    return R.astype(np.float32)

def robust_z(A):
    med = np.median(A, 0)
    mad = 1.4826 * np.median(np.abs(A - med), 0)
    sd = A.std(0)
    scale = np.where(mad > 1e-3, mad, sd + 1e-6)      # cohort-level fallback for markers with MAD = 0
    return np.clip((A - med) / scale, -5, 5).astype(np.float32)

def load(key, P):
    report = json.load(open(P["report"]))
    cells = pd.read_parquet(P["cells"])
    if key == "danenberg":
        chan = list(report["channel_map"].values())
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid, n_prot = "metabric_id", 36
    else:
        chan = list(report["channel_map"])
        cells = cells[cells.is_tumour == 1]
        pid, n_prot = "patient_id", 32
    protein = [c for c in chan if c not in ("histone_h3", "dna1", "dna2")]
    assert len(protein) == n_prot
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    epi = (cells.is_epithelial.values == 1) if key == "danenberg" else (cells.broad_class.values == "epithelial")
    img_code = pd.factorize(cells.image_name, sort=True)[0]
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    A = np.arcsinh(cells[protein].values.astype(np.float64) / COFACTOR)
    Z_img = np.empty_like(A, dtype=np.float32)
    for a, b in zip(starts[:-1], starts[1:]):             # identical to the earlier runs
        med = np.median(A[a:b], 0)
        mad = np.median(np.abs(A[a:b] - med), 0)
        Z_img[a:b] = np.clip((A[a:b] - med) / (1.4826 * mad + 1e-6), -5, 5)
    Z_coh = robust_z(A)
    spaces = {"int_img": Z_img, "rank_img": rank_vec(Z_img), "int_coh": Z_coh, "rank_coh": rank_vec(Z_coh)}
    patient = cells[pid].astype(str).values
    print(f"{key}: {len(cells):,} cells, {len(starts) - 1} images, {len(np.unique(patient))} patients, "
          f"{epi.mean():.2f} epithelial")
    return dict(protein=protein, A=A.astype(np.float32), spaces=spaces, patient=patient, epi=epi,
                phen=cells.cell_phenotype.astype(str).values)

def clinical_danenberg(P):
    c = pd.read_parquet(P["clinical"]).replace({"nan": np.nan})
    c["pid"] = c.metabric_id.astype(str)
    c = c.set_index("pid")
    split = pd.read_csv(P["split"]).set_index("metabric_id")
    t = pd.DataFrame(index=c.index)
    t["pam50"] = c.CLAUDIN_SUBTYPE.fillna(pd.Series(c.index, index=c.index).map(split.pam50))
    t["intclust"] = c.INTCLUST
    t["er"], t["pr"], t["her2"] = c.ER_STATUS, c.PR_STATUS, c.HER2_STATUS
    g = num(c.GRADE).values
    t["grade3"] = np.where(np.isnan(g), None, np.where(g == 3, "3", "1-2"))
    ln = num(c.LYMPH_NODES_EXAMINED_POSITIVE).values
    t["ln_pos"] = np.where(np.isnan(ln), None, np.where(ln > 0, "pos", "neg"))
    vs = c.VITAL_STATUS
    t["dss_time"] = num(c.OS_MONTHS).values
    t["dss_event"] = np.where(vs.isna(), np.nan, (vs == "Died of Disease").astype(float))
    rs = c.RFS_STATUS
    t["rfs_time"] = num(c.RFS_MONTHS).values
    t["rfs_event"] = np.where(rs.isna(), np.nan, rs.astype(str).str.startswith("1").astype(float))
    cov = pd.DataFrame(index=c.index)
    cov["age"] = num(c.AGE_AT_DIAGNOSIS).values
    cov["grade"] = g
    cov["log_size"] = np.log1p(num(c.TUMOR_SIZE).values)
    cov["log_nodes"] = np.log1p(ln)
    cov["er_pos"] = np.where(c.ER_STATUS.isna(), np.nan, (c.ER_STATUS == "Positive").astype(float))
    return t, cov, {"dss": ("dss_time", "dss_event"), "rfs": ("rfs_time", "rfs_event")}

def clinical_jackson(P):
    c = pd.read_parquet(P["clinical"]).replace({"nan": np.nan, "[]": np.nan})
    c = c.groupby("patient_id").first()
    c.index = c.index.astype(str)
    t = pd.DataFrame(index=c.index)
    t["clinical_type"] = c.clinical_type
    g = num(c.grade).values                                # METASTASIS -> NaN
    t["grade3"] = np.where(np.isnan(g), None, np.where(g == 3, "3", "1-2"))
    t["er"], t["pr"], t["her2"] = c.ERStatus, c.PRStatus, c.HER2Status
    basel = (c.cohort == "basel").values
    st = c.Patientstatus
    ok = basel & st.notna().values
    t["dss_time"] = np.where(basel, num(c.OSmonth).values, np.nan)
    t["dss_event"] = np.where(ok, (st == "death by primary disease").astype(float), np.nan)
    t["os_time"] = t.dss_time
    t["os_event"] = np.where(ok, st.isin(["death", "death by primary disease"]).astype(float), np.nan)
    cov = pd.DataFrame(index=c.index)
    cov["age"] = num(c.age).values
    cov["grade"] = g
    cov["log_size"] = np.log1p(num(c.tumor_size).values)
    n = c.PTNM_N.astype(str).str.strip().str.lower().str.lstrip("n")      # Basel "0", "1a"; Zurich "n0", "n1"
    cov["n_pos"] = np.where(n.str.startswith("0"), 0.0, np.where(n.str.match(r"^[123]"), 1.0, np.nan))
    return t, cov, {"dss": ("dss_time", "dss_event"), "os": ("os_time", "os_event")}

## Load both cohorts

In [ ]:
PATHS = locate()
DATA, CLIN = {}, {}
for key in ("danenberg", "jackson"):
    DATA[key] = load(key, PATHS[key])
    CLIN[key] = (clinical_danenberg if key == "danenberg" else clinical_jackson)(PATHS[key])
    t = CLIN[key][0]
    pats = np.unique(DATA[key]["patient"])
    print(key, "patients with clinical:", t.index.isin(pats).sum(), "of", len(pats))
    print(t.reindex(pats).notna().sum().to_string())

## Cell states, patient features, tumour-state entropy, patient-specificity (M1)

In [ ]:
def shares(labels, pat_code, n_pat, k):
    M = np.zeros((n_pat, k))
    ok = labels >= 0
    np.add.at(M, (pat_code[ok], labels[ok]), 1)
    return M

def clr(M, eps=0.5):
    L = np.log(M + eps)                    # pseudo-count on counts, then share-invariant CLR
    return L - L.mean(1, keepdims=True)

def entropy_rows(M):
    p = M / np.maximum(M.sum(1, keepdims=True), 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -(np.where(p > 0, p * np.log(p), 0)).sum(1)
    return h

def fit_states(V, k, seed, idx):
    rng = np.random.default_rng(seed)
    fit_idx = rng.choice(idx, min(N_FIT, len(idx)), replace=False)
    km = MiniBatchKMeans(k, random_state=seed, batch_size=4096, n_init=3, max_iter=200).fit(V[fit_idx])
    lab = np.full(len(V), -1)
    lab[idx] = km.predict(V[idx])
    return lab

def patient_specificity(lab, pat_code, k):
    """Share of cells that sit in states where a single patient contributes > 50% of the cells."""
    ok = lab >= 0
    ct = pd.crosstab(lab[ok], pat_code[ok])
    top = ct.max(1) / ct.sum(1)
    private = top.index[top > 0.5]
    eff = np.exp(entropy_rows(ct.values))                  # effective number of patients per state
    return float(np.isin(lab[ok], private).mean()), float(np.median(eff))

FEAT, TUMH, MECH = {}, {}, []
t0 = time.time()
for key, D in DATA.items():
    pats, pat_code = np.unique(D["patient"], return_inverse=True)
    n_pat = len(pats)
    all_idx, epi_idx = np.arange(len(pat_code)), np.flatnonzero(D["epi"])
    n_tum = np.bincount(pat_code[epi_idx], minlength=n_pat)
    FEAT[key], TUMH[key] = {"pats": pats, "n_tum": n_tum}, {}
    for sp in SPACES:
        V = D["spaces"][sp]
        for k in KS_ALL:
            for s in SEEDS:
                lab = fit_states(V, k, s, all_idx)
                FEAT[key][(sp, k, s)] = clr(shares(lab, pat_code, n_pat, k))
                ps, eff = patient_specificity(lab, pat_code, k)
                MECH.append(dict(cohort=key, space=sp, k=k, seed=s, share_cells_private_states=ps,
                                 median_patients_per_state=eff))
        for k in KS_TUM:
            for s in SEEDS:
                lab = fit_states(V, k, s, epi_idx)
                TUMH[key][(sp, k, s)] = entropy_rows(shares(lab, pat_code, n_pat, k))
        print(f"{key} {sp}: {(time.time() - t0) / 60:.1f} min")
    ph_code, ph_names = pd.factorize(D["phen"])
    FEAT[key][("phenotypes", 0, 0)] = clr(shares(ph_code, pat_code, n_pat, len(ph_names)))
    PB = np.zeros((n_pat, D["A"].shape[1]))
    np.add.at(PB, pat_code, D["A"])
    FEAT[key][("pseudobulk", 0, 0)] = PB / np.bincount(pat_code, minlength=n_pat)[:, None]
    epi_ph = np.where(D["epi"], ph_code, -1)
    TUMH[key][("phenotypes", 0, 0)] = entropy_rows(shares(epi_ph, pat_code, n_pat, len(ph_names)))
mech = pd.DataFrame(MECH)
mech.to_csv(f"{OUT}/state_patient_specificity.csv", index=False)
print(mech.groupby(["cohort", "space", "k"])[["share_cells_private_states", "median_patients_per_state"]].mean().round(3))

## T1: classification

In [ ]:
def groups_of(key):
    """Feature-set groups: (space, k) -> list of seed keys; reference sets have a single key."""
    g = {}
    for fk in FEAT[key]:
        if isinstance(fk, tuple):
            g.setdefault(fk[:2], []).append(fk)
    return g

def auc(y, P, classes):
    if len(classes) == 2:
        return roc_auc_score(y == classes[1], P[:, 1])
    return roc_auc_score(y, P, multi_class="ovr", average="macro", labels=classes)

def oof_class(X, y):
    classes = np.unique(y)
    P = np.zeros((len(y), len(classes)))
    for r in range(N_REP):
        for tr, te in StratifiedKFold(N_FOLD, shuffle=True, random_state=SEED + r).split(X, y):
            m = make_pipeline(StandardScaler(), LogisticRegression(C=C_LOGIT, max_iter=5000))
            m.fit(X[tr], y[tr])
            P[te] += m.predict_proba(X[te])
    return P / N_REP

CLASS_TARGETS = {"danenberg": ["pam50", "intclust", "er", "pr", "her2", "grade3", "ln_pos"],
                 "jackson": ["clinical_type", "grade3", "er", "pr", "her2"]}
jobs, meta = [], []
YS = {}
for key in DATA:
    t = CLIN[key][0].reindex(FEAT[key]["pats"])
    for tg in CLASS_TARGETS[key]:
        y = t[tg].astype(object).where(t[tg].notna(), None).values
        keep = np.array([v is not None for v in y])
        vc = pd.Series(y[keep]).value_counts()
        keep &= np.isin(y, vc.index[vc >= MIN_CLASS])
        yk = np.array([str(v) for v in y[keep]])
        if len(np.unique(yk)) < 2:
            print(f"skip {key} {tg}: fewer than 2 classes")
            continue
        YS[(key, tg)] = (keep, yk, np.unique(yk))
        for grp, fks in groups_of(key).items():
            for fk in fks:
                jobs.append(delayed(oof_class)(FEAT[key][fk][keep], yk))
                meta.append((key, tg, grp, fk))
t0 = time.time()
res = Parallel(n_jobs=N_JOBS, verbose=1)(jobs)
print(f"classification: {(time.time() - t0) / 60:.1f} min, {len(jobs)} feature sets")
PRED = {}
for (key, tg, grp, fk), P in zip(meta, res):
    PRED.setdefault((key, tg, grp), []).append(P)
PRED = {k: np.mean(v, 0) for k, v in PRED.items()}           # average over k-means seeds

def boot_pair(fn, n, *arrs):
    r = np.random.default_rng(SEED)
    m = len(arrs[0]); out = []
    for _ in range(n):
        i = r.integers(0, m, m)
        try:
            out.append(fn(*[a[i] for a in arrs]))
        except ValueError:
            continue
    return np.percentile(out, [2.5, 97.5]) if out else (np.nan, np.nan)

rows = []
for (key, tg, grp), P in PRED.items():
    keep, yk, classes = YS[(key, tg)]
    per_seed = [auc(yk, p, classes) for (kk, tt, gg, fk), p in zip(meta, res) if (kk, tt, gg) == (key, tg, grp)]
    rows.append(dict(cohort=key, target=tg, space=grp[0], k=grp[1], n=len(yk), n_classes=len(classes),
                     auc=auc(yk, P, classes), auc_seed_min=min(per_seed), auc_seed_max=max(per_seed)))
cls = pd.DataFrame(rows)
diff = []
for key in DATA:
    for tg in CLASS_TARGETS[key]:
        if (key, tg) not in YS:
            continue
        keep, yk, classes = YS[(key, tg)]
        for a, b in PAIRS:
            for k in KS_ALL:
                Pa, Pb = PRED[(key, tg, (a, k))], PRED[(key, tg, (b, k))]
                d = auc(yk, Pb, classes) - auc(yk, Pa, classes)
                lo, hi = boot_pair(lambda y, pa, pb: auc(y, pb, classes) - auc(y, pa, classes), N_BOOT, yk, Pa, Pb)
                diff.append(dict(cohort=key, target=tg, norm=a.split("_")[1], k=k, auc_intensity=auc(yk, Pa, classes),
                                 auc_rank=auc(yk, Pb, classes), rank_minus_int=d, lo=lo, hi=hi, n=len(yk)))
cls_diff = pd.DataFrame(diff)
cls.to_csv(f"{OUT}/utility_classification.csv", index=False)
cls_diff.to_csv(f"{OUT}/utility_classification_rank_vs_intensity.csv", index=False)
print(cls.pivot_table(index=["cohort", "target"], columns=["space", "k"], values="auc").round(3).to_string())
print(cls_diff.round(3).to_string(index=False))

## T2: survival

In [ ]:
def oof_surv(X, C, T, E):
    risk = np.zeros(len(T))
    for r in range(N_REP):
        for tr, te in StratifiedKFold(N_FOLD, shuffle=True, random_state=SEED + r).split(T, E):
            ptr, pte = [], []
            if X is not None:
                sc = StandardScaler().fit(X[tr])
                pca = PCA(min(N_PCA, X.shape[1]), random_state=0).fit(sc.transform(X[tr]))
                ptr.append(pca.transform(sc.transform(X[tr]))); pte.append(pca.transform(sc.transform(X[te])))
            if C is not None:
                sc2 = StandardScaler().fit(C[tr])
                ptr.append(sc2.transform(C[tr])); pte.append(sc2.transform(C[te]))
            Xtr, Xte = np.hstack(ptr), np.hstack(pte)
            df = pd.DataFrame(Xtr, columns=[f"x{j}" for j in range(Xtr.shape[1])])
            df["T"], df["E"] = T[tr], E[tr]
            try:
                beta = CoxPHFitter(penalizer=COX_PEN).fit(df, "T", "E").params_.values
            except Exception:
                beta = CoxPHFitter(penalizer=1.0).fit(df, "T", "E").params_.values
            h = Xte @ beta
            risk[te] += (h - h.mean()) / (h.std() + 1e-12)
    return risk / N_REP

def cidx(T, E, risk):
    return concordance_index(T, -risk, E)

SURV_DATA = {}
jobs, meta = [], []
for key in DATA:
    t, cov, surv = CLIN[key]
    t, cov = t.reindex(FEAT[key]["pats"]), cov.reindex(FEAT[key]["pats"])
    for sname, (tc, ec) in surv.items():
        keep = t[tc].notna().values & t[ec].notna().values & cov.notna().all(1).values
        T = np.maximum(t[tc].values[keep].astype(float), 0.1)
        E = t[ec].values[keep].astype(float)
        C = cov.values[keep].astype(float)
        print(f"{key} {sname}: {keep.sum()} patients, {int(E.sum())} events")
        assert keep.sum() >= 50, (key, sname, cov.notna().sum().to_dict())
        SURV_DATA[(key, sname)] = (keep, T, E, C)
        jobs.append(delayed(oof_surv)(None, C, T, E)); meta.append((key, sname, ("clinical", 0), None, "clinical"))
        for grp, fks in groups_of(key).items():
            for fk in fks:
                X = FEAT[key][fk][keep]
                jobs.append(delayed(oof_surv)(X, None, T, E)); meta.append((key, sname, grp, fk, "features"))
                jobs.append(delayed(oof_surv)(X, C, T, E)); meta.append((key, sname, grp, fk, "features+clinical"))
t0 = time.time()
res = Parallel(n_jobs=N_JOBS, verbose=1)(jobs)
print(f"survival: {(time.time() - t0) / 60:.1f} min, {len(jobs)} fits")
RISK = {}
for (key, sname, grp, fk, mode), r in zip(meta, res):
    RISK.setdefault((key, sname, grp, mode), []).append(r)
RISK = {k: np.mean(v, 0) for k, v in RISK.items()}
rows = []
for (key, sname, grp, mode), r in RISK.items():
    keep, T, E, C = SURV_DATA[(key, sname)]
    rows.append(dict(cohort=key, endpoint=sname, space=grp[0], k=grp[1], mode=mode, n=len(T), events=int(E.sum()),
                     c_index=cidx(T, E, r)))
surv = pd.DataFrame(rows)
diff = []
for key in DATA:
    for sname in CLIN[key][2]:
        keep, T, E, C = SURV_DATA[(key, sname)]
        base = RISK[(key, sname, ("clinical", 0), "clinical")]
        for mode in ("features", "features+clinical"):
            for a, b in PAIRS:
                for k in KS_ALL:
                    ra, rb = RISK[(key, sname, (a, k), mode)], RISK[(key, sname, (b, k), mode)]
                    lo, hi = boot_pair(lambda t_, e_, x, y: cidx(t_, e_, y) - cidx(t_, e_, x), N_BOOT, T, E, ra, rb)
                    diff.append(dict(cohort=key, endpoint=sname, mode=mode, norm=a.split("_")[1], k=k,
                                     c_intensity=cidx(T, E, ra), c_rank=cidx(T, E, rb),
                                     rank_minus_int=cidx(T, E, rb) - cidx(T, E, ra), lo=lo, hi=hi,
                                     c_clinical_only=cidx(T, E, base), n=len(T), events=int(E.sum())))
surv_diff = pd.DataFrame(diff)
surv.to_csv(f"{OUT}/utility_survival.csv", index=False)
surv_diff.to_csv(f"{OUT}/utility_survival_rank_vs_intensity.csv", index=False)
print(surv.pivot_table(index=["cohort", "endpoint", "mode"], columns=["space", "k"], values="c_index").round(3).to_string())
print(surv_diff.round(3).to_string(index=False))

## T3: tumour-cell heterogeneity and DSS

In [ ]:
def cox_hr(h, C, T, E):
    df = pd.DataFrame(C, columns=[f"c{j}" for j in range(C.shape[1])])
    df["h"] = (h - h.mean()) / (h.std() + 1e-12)
    df["T"], df["E"] = T, E
    cph = CoxPHFitter(penalizer=0.01).fit(df, "T", "E")
    return cph.params_["h"], cph.standard_errors_["h"], cph.summary.loc["h", "p"]

def het_group(key):
    g = {}
    for fk in TUMH[key]:
        g.setdefault(fk[:2], []).append(fk)
    return g

rows, diffs = [], []
for key in DATA:
    t, cov, surv_def = CLIN[key]
    t, cov = t.reindex(FEAT[key]["pats"]), cov.reindex(FEAT[key]["pats"])
    ok_tum = FEAT[key]["n_tum"] >= MIN_TUM
    sub_col = "pam50" if key == "danenberg" else "clinical_type"
    H = {grp: np.mean([TUMH[key][fk] for fk in fks], 0) for grp, fks in het_group(key).items()}
    tc, ec = surv_def["dss"]
    keep = ok_tum & t[tc].notna().values & t[ec].notna().values & cov.notna().all(1).values
    T = np.maximum(t[tc].values[keep].astype(float), 0.1); E = t[ec].values[keep].astype(float)
    C = np.c_[cov.values[keep].astype(float), np.log(FEAT[key]["n_tum"][keep])]
    g = cov["grade"].values
    for grp, h in H.items():
        b, se, p = cox_hr(h[keep], C, T, E)
        m = ok_tum & ~np.isnan(g)
        rho = spearmanr(h[m], g[m])[0]
        s = t[sub_col].values
        ms = ok_tum & pd.notna(s)
        groups = [h[ms][s[ms] == v] for v in pd.unique(s[ms]) if (s[ms] == v).sum() >= MIN_CLASS]
        kw = kruskal(*groups)
        n_kw = sum(len(x) for x in groups)
        rows.append(dict(cohort=key, space=grp[0], k=grp[1], n_dss=int(keep.sum()), events=int(E.sum()),
                         log_hr_per_sd=b, hr_per_sd=np.exp(b), hr_lo=np.exp(b - 1.96 * se), hr_hi=np.exp(b + 1.96 * se),
                         p=p, spearman_grade=rho, kruskal_subtype_H=kw.statistic, kruskal_p=kw.pvalue,
                         subtype_eta2=(kw.statistic - len(groups) + 1) / (n_kw - len(groups)),
                         mean_entropy=float(np.nanmean(h[ok_tum]))))
    for a, b_ in PAIRS:
        for k in KS_TUM:
            ha, hb = H[(a, k)][keep], H[(b_, k)][keep]
            d = cox_hr(hb, C, T, E)[0] - cox_hr(ha, C, T, E)[0]
            r = np.random.default_rng(SEED); bs = []
            for _ in range(N_BOOT_HR):
                i = r.integers(0, len(T), len(T))
                try:
                    bs.append(cox_hr(hb[i], C[i], T[i], E[i])[0] - cox_hr(ha[i], C[i], T[i], E[i])[0])
                except Exception:
                    continue
            diffs.append(dict(cohort=key, norm=a.split("_")[1], k=k, log_hr_rank_minus_int=d,
                              lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5),
                              corr_entropy_rank_vs_int=spearmanr(ha, hb)[0]))
het = pd.DataFrame(rows)
het_diff = pd.DataFrame(diffs)
het.to_csv(f"{OUT}/heterogeneity_dss.csv", index=False)
het_diff.to_csv(f"{OUT}/heterogeneity_dss_rank_vs_intensity.csv", index=False)
print(het.round(3).to_string(index=False))
print(het_diff.round(3).to_string(index=False))

## Pre-registered answers

In [ ]:
def pick(df, **kw):
    m = np.ones(len(df), bool)
    for c, v in kw.items():
        m &= (df[c] == v).values
    return df[m]

ans = {}
pc = pick(cls_diff, norm="img", k=20)
d_pos = {r.target: bool(r.lo > 0) for r in pick(pc, cohort="danenberg").itertuples()}
j_sign = {r.target: float(np.sign(r.rank_minus_int)) for r in pick(pc, cohort="jackson").itertuples()}
ps = pick(surv_diff, norm="img", k=20, endpoint="dss", mode="features+clinical")
dss_pos = {r.cohort: bool(r.lo > 0) for r in ps.itertuples()}
ph = pick(het_diff, norm="img", k=10)
ans["T1_danenberg_targets_rank_better_CI"] = d_pos
ans["T1_jackson_sign"] = j_sign
ans["T1_rule_met"] = bool(sum(d_pos.values()) >= len(d_pos) / 2 and
                          all(j_sign.get(t, 1) > 0 for t, v in d_pos.items() if v))
ans["T2_dss_features+clinical_rank_better_CI"] = dss_pos
ans["T2_rule_met"] = bool(dss_pos and all(dss_pos.values()))
ans["utility_shown"] = ans["T1_rule_met"] or ans["T2_rule_met"]
hd = pick(ph, cohort="danenberg").iloc[0]; hj = pick(ph, cohort="jackson").iloc[0]
ans["T3_log_hr_rank_minus_int"] = {"danenberg": [hd.log_hr_rank_minus_int, hd.lo, hd.hi],
                                   "jackson": [hj.log_hr_rank_minus_int, hj.lo, hj.hi]}
hr = pick(het, space="rank_img", k=10); hi_ = pick(het, space="int_img", k=10)
ans["T3_hr_per_sd"] = {c: {"intensity": float(pick(hi_, cohort=c).hr_per_sd.iloc[0]),
                           "rank": float(pick(hr, cohort=c).hr_per_sd.iloc[0])} for c in DATA}
bigger = [np.abs(pick(hr, cohort=c).log_hr_per_sd.iloc[0]) > np.abs(pick(hi_, cohort=c).log_hr_per_sd.iloc[0]) for c in DATA]
ans["T3_rule_met"] = bool((hd.lo > 0 or hd.hi < 0) and bigger[0] and bigger[1])
ans["M1_private_state_share"] = mech.groupby(["cohort", "space"]).share_cells_private_states.mean().round(3).unstack().to_dict()
json.dump(ans, open(f"{OUT}/summary_utility.json", "w"), indent=1, default=float)
print(json.dumps(ans, indent=1, default=float))